In [1]:
import torch
import ultralytics

print("--- SYSTEM DIAGNOSTICS ---")
print(f"PyTorch Version: {torch.__version__}")
print(f"Ultralytics Version: {ultralytics.__version__}\n")

cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
else:
    print("\n⚠️ WARNING: CUDA is NOT available!")
    print("Your models will train on the CPU, which is extremely slow.")

--- SYSTEM DIAGNOSTICS ---
PyTorch Version: 2.7.1+cu118
Ultralytics Version: 8.4.37

CUDA Available: True
GPU Device Name: NVIDIA GeForce RTX 3060
Number of GPUs: 1


In [12]:
import os

# --- UPDATE THIS TO YOUR MAIN RAF-DB FOLDER ---
# (The folder that contains the 'train' and 'test' subfolders)
DATASET_DIR = r'C:\Users\User\Desktop\YoloTraining\RAF-DB'
# ----------------------------------------------

emotion_map = {
    '1': 'surprise', '2': 'fear', '3': 'disgust', 
    '4': 'happy', '5': 'sad', '6': 'angry', '7': 'neutral'
}

print("Renaming folders...")

for split in ['train', 'test']:
    split_dir = os.path.join(DATASET_DIR, split)
    
    # Check if the train/test folder exists
    if not os.path.exists(split_dir):
        print(f"⚠️ Could not find folder: {split_dir}")
        continue

    for folder_num, emotion_name in emotion_map.items():
        old_path = os.path.join(split_dir, folder_num)
        new_path = os.path.join(split_dir, emotion_name)
        
        # If the numbered folder exists, rename it
        if os.path.exists(old_path):
            os.rename(old_path, new_path)
            print(f"✅ Renamed: {split}/{folder_num} -> {split}/{emotion_name}")
        elif os.path.exists(new_path):
            print(f"ℹ️ Already named correctly: {split}/{emotion_name}")

print("\n🚀 Phase 1 Complete! Move straight to the Phase 2 Preprocessing script.")

Renaming folders...
✅ Renamed: train/1 -> train/surprise
✅ Renamed: train/2 -> train/fear
✅ Renamed: train/3 -> train/disgust
✅ Renamed: train/4 -> train/happy
✅ Renamed: train/5 -> train/sad
✅ Renamed: train/6 -> train/angry
✅ Renamed: train/7 -> train/neutral
✅ Renamed: test/1 -> test/surprise
✅ Renamed: test/2 -> test/fear
✅ Renamed: test/3 -> test/disgust
✅ Renamed: test/4 -> test/happy
✅ Renamed: test/5 -> test/sad
✅ Renamed: test/6 -> test/angry
✅ Renamed: test/7 -> test/neutral

🚀 Phase 1 Complete! Move straight to the Phase 2 Preprocessing script.


In [13]:
from ultralytics import YOLO
import os

DATA_PATH = r"C:\Users\User\Desktop\YoloTraining\RAF-DB"

# 1. Load the MEDIUM classification model
model = YOLO('yolov8m-cls.pt') 

# 2. Start Training (Removed image_weights)
results = model.train(
    data=DATA_PATH, 
    epochs=50, 
    imgsz=224, 
    batch=16,
    # image_weights=True,  <-- REMOVED because it's for detection only
    project='Emotion_Project',
    name='YOLO_Medium_Raw_Run'
)

New https://pypi.org/project/ultralytics/8.4.38 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.37  Python-3.10.20 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\User\Desktop\YoloTraining\RAF-DB, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m-cls.pt, momentum=0.937, mosaic=

In [14]:
import os
from ultralytics import YOLO
from sklearn.metrics import classification_report

# 1. Load your freshly trained model
# Make sure this points to the "best.pt" file YOLO just generated
model_path = r"C:\Users\User\Desktop\YoloTraining\runs\classify\Emotion_Project\YOLO_Medium_Raw_Run\weights\best.pt"
model = YOLO(model_path)

# 2. Point to your validation folder
val_dir = r"C:\Users\User\Desktop\YoloTraining\RAF-DB\test"

# YOLO automatically saved your class names inside the model
class_names = model.names # Looks like {0: 'angry', 1: 'disgust', ...}
target_names = [class_names[i] for i in range(len(class_names))]

y_true = []
y_pred = []

print("Running YOLO on validation images... (This will take about 15-30 seconds)")

# 3. Loop through the validation images and make predictions
for class_id, class_name in class_names.items():
    folder_path = os.path.join(val_dir, class_name)
    if not os.path.isdir(folder_path): continue
    
    for img_name in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_name)
        
        # Make a prediction (verbose=False stops it from spamming your console)
        results = model(img_path, verbose=False)
        
        # Extract the highest probability guess
        pred_class_id = results[0].probs.top1
        
        # Record the true answer and YOLO's guess
        y_true.append(class_id)
        y_pred.append(pred_class_id)

# 4. Print the final table!
print("\n" + "="*55)
print(" YOLOv8 CLASSIFICATION REPORT (BASELINE) ")
print("="*55)
report = classification_report(y_true, y_pred, target_names=target_names, digits=2)
print(report)

Running YOLO on validation images... (This will take about 15-30 seconds)

 YOLOv8 CLASSIFICATION REPORT (BASELINE) 
              precision    recall  f1-score   support

       angry       0.81      0.83      0.82       162
     disgust       0.67      0.60      0.63       160
        fear       0.79      0.59      0.68        74
       happy       0.95      0.95      0.95      1185
     neutral       0.84      0.89      0.86       680
         sad       0.85      0.86      0.86       478
    surprise       0.90      0.86      0.88       329

    accuracy                           0.88      3068
   macro avg       0.83      0.80      0.81      3068
weighted avg       0.88      0.88      0.88      3068



In [15]:
import os
import shutil

train_path = r"C:\Users\User\Desktop\YoloTraining\RAF-DB\train"

# 1. Automatically find the largest folder's count (Happy = ~4700)
counts = []
for emotion in os.listdir(train_path):
    emotion_dir = os.path.join(train_path, emotion)
    if os.path.isdir(emotion_dir):
        counts.append(len(os.listdir(emotion_dir)))

target_count = max(counts)
print(f"Largest class found! Setting target count to: {target_count}")

# 2. Oversample the smaller classes to match
for emotion in os.listdir(train_path):
    emotion_dir = os.path.join(train_path, emotion)
    if not os.path.isdir(emotion_dir): continue
    
    images = [f for f in os.listdir(emotion_dir) if not f.startswith("aug_")]
    current_count = len(images)
    
    if current_count < target_count:
        print(f"Balancing {emotion}: {current_count} -> {target_count}")
        to_add = target_count - current_count
        
        # The Modulo loop handles the 16x multiplier effortlessly
        for i in range(to_add):
            source_img = images[i % current_count] 
            src_path = os.path.join(emotion_dir, source_img)
            dst_path = os.path.join(emotion_dir, f"aug_{i}_{source_img}")
            shutil.copy(src_path, dst_path)

print("Oversampling complete! You are ready to train.")

Largest class found! Setting target count to: 4772
Balancing angry: 705 -> 4772
Balancing disgust: 717 -> 4772
Balancing fear: 281 -> 4772
Balancing neutral: 2524 -> 4772
Balancing sad: 1982 -> 4772
Balancing surprise: 1290 -> 4772
Oversampling complete! You are ready to train.


In [16]:
from ultralytics import YOLO

# 1. Point this to your main RAF-DB folder
DATA_PATH = r"C:\Users\User\Desktop\YoloTraining\RAF-DB"

# 2. Load the YOLO Medium classification model
model = YOLO('yolov8m-cls.pt') 

# 3. Start Training 
# Because you have ~32,000 images now (7 classes * ~4700), this will take longer than the first run!
results = model.train(
    data=DATA_PATH, 
    epochs=50, 
    imgsz=224, 
    batch=16, 
    project='Emotion_Project',
    name='YOLO_Medium_Oversampled_Run' 
)

print("Oversampled Training Complete!")

New https://pypi.org/project/ultralytics/8.4.38 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.37  Python-3.10.20 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\User\Desktop\YoloTraining\RAF-DB, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m-cls.pt, momentum=0.937, mosaic=

In [17]:
import os
from ultralytics import YOLO
from sklearn.metrics import classification_report

# 1. Load the new Oversampled model
model_path = r"C:\Users\User\Desktop\YoloTraining\runs\classify\Emotion_Project\YOLO_Medium_Oversampled_Run\weights\best.pt"
model = YOLO(model_path)

# 2. Point to your validation/test folder
val_dir = r"C:\Users\User\Desktop\YoloTraining\RAF-DB\test"

# Since you renamed the folders before training, YOLO saved the English names!
class_names = model.names 
target_names = [class_names[i] for i in range(len(class_names))]

y_true = []
y_pred = []

print("Evaluating Oversampled Model... (This will take about 15-30 seconds)")

# 3. Run predictions
for class_id, class_name in class_names.items():
    folder_path = os.path.join(val_dir, class_name)
    if not os.path.isdir(folder_path): continue
    
    for img_name in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_name)
        
        results = model(img_path, verbose=False)
        pred_class_id = results[0].probs.top1
        
        y_true.append(class_id)
        y_pred.append(pred_class_id)

# 4. Print results
print("\n" + "="*55)
print(" YOLOv8 CLASSIFICATION REPORT (OVERSAMPLED) ")
print("="*55)
report = classification_report(y_true, y_pred, target_names=target_names, digits=2)
print(report)

Evaluating Oversampled Model... (This will take about 15-30 seconds)

 YOLOv8 CLASSIFICATION REPORT (OVERSAMPLED) 
              precision    recall  f1-score   support

       angry       0.84      0.86      0.85       162
     disgust       0.73      0.66      0.70       160
        fear       0.77      0.59      0.67        74
       happy       0.96      0.93      0.94      1185
     neutral       0.84      0.89      0.86       680
         sad       0.83      0.88      0.85       478
    surprise       0.88      0.86      0.87       329

    accuracy                           0.88      3068
   macro avg       0.83      0.81      0.82      3068
weighted avg       0.88      0.88      0.88      3068

